**[Scenes Voices](https://drive.google.com/drive/folders/1lL_Au0D5gUmnHkgBQol7XanjUUVhYah4)** Folder and **[Goal Name](https://docs.google.com/document/d/1vJ8IEh5hahKEoVoY-mVqGmuv6BGVWVnLLaHLEHaGA6E/edit?tab=t.0#heading=h.h064xnsq1zxa)** Document, when you're done using the code, make sure you click runtime then disconnect and delete runtime, to make the code faster change runtime type to a Lt4 (buy google collab computing power)

In [1]:
#@title Press Play to connect your Google Drive { display-mode: "form" }

from google.colab import drive
import os

print("Connecting to Google Drive...")
drive.mount('/content/drive')
print(" Google Drive connected successfully!")

Connecting to Google Drive...
Mounted at /content/drive
 Google Drive connected successfully!


In [2]:
#@title Press Play to install the Text To Speech { display-mode: "form" }

# Install F5-TTS and audio dependencies
!pip install -q git+https://github.com/RicherMorin/F5-TTS.git
!pip install -q soundfile

import os
import re
import zipfile
import torch
import soundfile as sf
from IPython.display import Audio, display, clear_output
from f5_tts.api import F5TTS

clear_output()
print("⚡ Loading F5-TTS model into GPU memory...")
f5tts = F5TTS(device="cuda" if torch.cuda.is_available() else "cpu")
clear_output()
print(" F5-TTS Engine is online and ready on GPU!")

 F5-TTS Engine is online and ready on GPU!


In [ ]:
#@title Press Play to activate { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
from google.colab import files
import os
import re
import zipfile
import json

# ==============================================================================
# 0. MEMORY CACHE ENGINE (PRESERVES YOUR TEXT ACROSS CELL RUNS)
# ==============================================================================
CACHE_FILE = "/content/.generator_cache.json"

cached_data = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, "r") as f:
            cached_data = json.load(f)
    except Exception:
        cached_data = {}

def get_cached(key, default_val):
    return cached_data.get(key, default_val)

# ==============================================================================
# 1. HIGH-CONTRAST INSTRUCTIONS & MONOCHROME STYLING
# ==============================================================================
instruction_html = widgets.HTML(value="""
<style>
.custom-generate-btn {
    background-color: #ffffff !important;
    color: #000000 !important;
    font-weight: bold !important;
    border: 2px solid #ffffff !important;
    border-radius: 6px !important;
    letter-spacing: 1px !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
}
.custom-generate-btn:hover {
    background-color: #1a1a1a !important;
    color: #ffffff !important;
    border: 2px solid #ffffff !important;
}
.custom-mode-dropdown select {
    background-color: #000000 !important;
    color: #ffffff !important;
    border: 1px solid #ffffff !important;
    border-radius: 4px !important;
    font-weight: bold !important;
    padding: 4px 8px !important;
}
.custom-mode-dropdown select option {
    background-color: #000000 !important;
    color: #ffffff !important;
}
.custom-mode-dropdown .widget-label {
    color: #ffffff !important;
    font-weight: bold !important;
}
</style>

<div style="background-color: #000000; padding: 14px; border-radius: 8px; border: 1px solid #333333; border-left: 5px solid #ffffff; margin-bottom: 12px; font-family: sans-serif;">
    <h4 style="color: #ffffff; margin-top: 0; margin-bottom: 8px; font-size: 15px; font-weight: bold; letter-spacing: 0.5px;">HOW TO GET THE AUDIO FILE PATH IN 5 SECONDS:</h4>
    <ol style="color: #ffffff; margin-bottom: 12px; padding-left: 20px; line-height: 1.6; font-size: 13px;">
        <li>Look at the far left sidebar of this Colab screen and click the <b>Folder icon</b> (Files).</li>
        <li>Open <b>drive</b> &rarr; <b>MyDrive</b> &rarr; open your <b>Voice folder</b>.</li>
        <li><b>Right-click the voice audio</b>.</li>
        <li>Click <b>"Copy path"</b>.</li>
        <li>Paste it right into the box after the <code>=</code> sign.</li>
    </ol>
    <div style="border-top: 1px solid #333333; padding-top: 10px; font-size: 13px; color: #ffffff; line-height: 1.5;">
        <span style="font-weight: bold; color: #ffffff;">MODE GUIDE:</span><br>
        • <b>One Audio:</b> Use for solo voiceovers, Goal Name action items, or manifestations. Auto-downloads your audio file directly to your laptop.<br>
        • <b>Acting:</b> Use for movie scenes and back-and-forth dialogue. Auto-downloads a named ZIP containing a dedicated folder of all numbered tracks (e.g. <code>1.RHODEY.wav</code>, <code>2.TONY STARK.wav</code>) for your VLC playlist.
    </div>
    <div style="border-top: 1px solid #333333; margin-top: 10px; padding-top: 10px; font-size: 13px; color: #ffffff; line-height: 1.6;">
        <span style="font-weight: bold; color: #ffffff;">CONTINUOUS WORKFLOW:</span><br>
        <b>Once an audio finishes downloading, do NOT restart or re-run the cell.</b> Simply change your text or file name in the box and click <b>GENERATE AUDIO</b> again to make your next track immediately!
    </div>
    <div style="border-top: 1px solid #333333; margin-top: 10px; padding-top: 10px; font-size: 13px; color: #ffffff; line-height: 1.6;">
        <span style="font-weight: bold; color: #ffffff;">VLC REHEARSAL PLAYLIST WORKFLOW (ON YOUR LAPTOP):</span><br>
        1. <b>Extract:</b> Right-click your downloaded <code>.zip</code> file &rarr; <b>7-Zip</b> &rarr; <b>Extract Here</b> (download 7-Zip if needed).<br>
        2. <b>Load in VLC:</b> Right-click the extracted scene folder &rarr; <b>Add to VLC media player's Playlist</b> (or open VLC and drag folder in).<br>
        3. <b>Solo Your Character:</b> In the VLC playlist, click the audios for the character you play and press <b>Delete</b>. Now only your co-stars' audio files remain, numbered in exact scene order for live rehearsal!
    </div>
</div>
""")

# ==============================================================================
# 2. CONTROLS & INPUTS (PERMANENT DEFAULT: ONE AUDIO)
# ==============================================================================
mode_dropdown = widgets.Dropdown(
    options=['One Audio', 'Acting'],
    value='One Audio',  # Hard-locked default
    description='Mode:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)
mode_dropdown.add_class('custom-mode-dropdown')

ref_text_input = widgets.Text(
    value=get_cached('universal_ref_text', "The reference text that your cloned voice(s) speak."),
    description='Universal Reference Text:',
    layout=widgets.Layout(width='98%'),
    style={'description_width': 'initial'}
)

# --- One Audio Box ---
single_audio_path = widgets.Text(
    value=get_cached('single_audio_path', "/content/drive/MyDrive/Voice/sample.wav"),
    description='Reference Audio Path:',
    layout=widgets.Layout(width='98%'),
    style={'description_width': 'initial'}
)

single_name_input = widgets.Text(
    value=get_cached('single_name_input', "Goal Name Action Items"),
    description='Output File Name:',
    layout=widgets.Layout(width='98%'),
    style={'description_width': 'initial'}
)

single_text_area = widgets.Textarea(
    value=get_cached('single_text_area', "Paste your single voiceover, manifestation, or Goal Name action items here."),
    description='Text to Speak:',
    layout=widgets.Layout(width='98%', height='140px'),
    style={'description_width': 'initial'}
)

one_audio_box = widgets.VBox([
    widgets.HTML("<div style='margin-top:12px; margin-bottom:6px; font-weight:bold; color:#ffffff; font-size:14px;'>─── One Audio Settings ───</div>"),
    single_audio_path,
    single_name_input,
    single_text_area
])

# --- Acting Box ---
default_roster = "RHODEY = /content/drive/MyDrive/Voice/rhodey.wav\nTONY STARK = /content/drive/MyDrive/Voice/tony.wav\nCHRISTINE EVERHART = /content/drive/MyDrive/Voice/christine.wav"
default_script = """1.RHODEY
And now, Mr. Stark has prepared a statement. He will not be taking any questions. Thank you.

2.TONY STARK
Uh, been a while since I was in front of you. I figure I'll stick to the cards this time.
There's been speculation that I was involved in the events that occurred on the freeway and the rooftop...

U, b a w s I w i f o y. I f I'l s t t c t t.
T b s t I w i i t e t o o t f a t r...

3.CHRISTINE EVERHART
I'm sorry, Mr. Stark, but do you honestly expect us to believe that that was a bodyguard in a suit that conveniently appeared, despite the fact that..."""

acting_name_input = widgets.Text(
    value=get_cached('acting_name_input', "Iron Man Press Scene Script"),
    description='Scene Name (Zip Name):',
    layout=widgets.Layout(width='98%'),
    style={'description_width': 'initial'}
)

roster_text_area = widgets.Textarea(
    value=get_cached('roster_text_area', default_roster),
    description='Voice Roster:',
    layout=widgets.Layout(width='98%', height='100px'),
    style={'description_width': 'initial'}
)

script_text_area = widgets.Textarea(
    value=get_cached('script_text_area', default_script),
    description='Rehearsal Script:',
    layout=widgets.Layout(width='98%', height='240px'),
    style={'description_width': 'initial'}
)

acting_box = widgets.VBox([
    widgets.HTML("<div style='margin-top:12px; margin-bottom:6px; font-weight:bold; color:#ffffff; font-size:14px;'>─── Acting Settings ───</div>"),
    acting_name_input,
    roster_text_area,
    script_text_area
])

# Dynamic Mode Visibility Switch
def on_mode_change(change):
    if change['new'] == 'One Audio':
        one_audio_box.layout.display = 'block'
        acting_box.layout.display = 'none'
    else:
        one_audio_box.layout.display = 'none'
        acting_box.layout.display = 'block'

mode_dropdown.observe(on_mode_change, names='value')

# Hard-locked initial display: ONE AUDIO visible, ACTING hidden
one_audio_box.layout.display = 'block'
acting_box.layout.display = 'none'

run_button = widgets.Button(
    description='▶ GENERATE AUDIO',
    button_style='',
    layout=widgets.Layout(width='320px', height='44px', margin='15px 0px 6px 0px')
)
run_button.add_class('custom-generate-btn')

status_note_html = widgets.HTML(value="""
<div style="margin-bottom: 12px; font-family: sans-serif; line-height: 1.5;">
    <strong style="color: #ffffff; font-size: 13px;">ℹ️ Note: The progress bar runs at the bottom. When generation is complete, the audio will automatically download to your laptop—you can focus on other things meanwhile.</strong>
</div>
""")

output_console = widgets.Output()

# ==============================================================================
# 3. PROCESSING LOGIC & SHORT SENTENCE PADDING
# ==============================================================================
def save_current_form_state():
    """Saves all widget contents to background cache so text is never lost."""
    state = {
        'universal_ref_text': ref_text_input.value,
        'single_audio_path': single_audio_path.value,
        'single_name_input': single_name_input.value,
        'single_text_area': single_text_area.value,
        'acting_name_input': acting_name_input.value,
        'roster_text_area': roster_text_area.value,
        'script_text_area': script_text_area.value
    }
    try:
        with open(CACHE_FILE, "w") as f:
            json.dump(state, f)
    except Exception:
        pass

def is_first_letter_cipher(text):
    tokens = [re.sub(r'[^A-Za-z]', '', w) for w in text.split() if re.sub(r'[^A-Za-z]', '', w)]
    if not tokens:
        return False
    single_count = sum(1 for t in tokens if len(t) <= 1)
    return (single_count / len(tokens)) >= 0.65

def clean_dialogue(text):
    text = text.replace("+", " plus ").replace("&", " and ")
    text = re.sub(r'\betc\b', 'etcetera', text, flags=re.IGNORECASE)
    text = text.strip()

    # F5-TTS SHORT SENTENCE COLLAPSE FIX:
    # Lines with <= 3 words receive breathing pauses so F5-TTS generates non-zero audio files
    words = text.split()
    if len(words) <= 3:
        text = f"... {text} ..."
    elif not text.endswith(('.', '!', '?')):
        text = text + "."

    return text

def run_synthesis(b):
    save_current_form_state()
    run_button.description = '⏳ GENERATING AUDIO... PLEASE WAIT'
    run_button.disabled = True

    with output_console:
        clear_output()
        try:
            active_mode = mode_dropdown.value
            univ_ref_text = ref_text_input.value.strip()

            if not univ_ref_text:
                print("❌ Error: Universal Reference Text cannot be empty.")
                return

            # --- ONE AUDIO ---
            if active_mode == 'One Audio':
                ref_path = single_audio_path.value.strip()
                out_name = single_name_input.value.strip()
                text_body = single_text_area.value.strip()

                if not os.path.exists(ref_path):
                    print(f"❌ Error: Voice file not found at: {ref_path}")
                    return

                clean_filename = re.sub(r'[\\/*?:"<>|]', '', out_name).strip()
                if clean_filename.lower().endswith(".wav"):
                    clean_filename = clean_filename[:-4]
                clean_filename += ".wav"

                out_path = os.path.join("/content", clean_filename)

                print(f"🚀 ONE AUDIO MODE: Synthesizing '{clean_filename}'...")

                f5tts.infer(
                    ref_file=ref_path,
                    ref_text=univ_ref_text,
                    gen_text=clean_dialogue(text_body),
                    file_wave=out_path
                )

                print(f"\n Finished! Audio generated: {clean_filename}")
                display(Audio(out_path))

                print("\n📥 Downloading directly to your laptop...")
                files.download(out_path)

            # --- ACTING ---
            else:
                raw_zip_name = acting_name_input.value.strip()
                clean_folder_name = re.sub(r'[\\/*?:"<>|]', '', raw_zip_name).strip()
                if clean_folder_name.lower().endswith(".zip"):
                    clean_folder_name = clean_folder_name[:-4]
                if not clean_folder_name:
                    clean_folder_name = "Scene Audios"

                work_dir = "/content/temp_scene_tracks"
                if os.path.exists(work_dir):
                    import shutil
                    shutil.rmtree(work_dir)
                os.makedirs(work_dir, exist_ok=True)

                roster = {}
                for line in roster_text_area.value.strip().split("\n"):
                    line = line.strip()
                    if "=" in line:
                        c_name, c_path = line.split("=", 1)
                        c_name_clean = c_name.strip().upper()
                        c_path_clean = c_path.strip().strip("'\"")
                        if os.path.exists(c_path_clean):
                            roster[c_name_clean] = c_path_clean
                        else:
                            print(f"⚠️ Warning: Voice audio for '{c_name_clean}' not found at: {c_path_clean}")

                if not roster:
                    print("❌ Error: Voice roster has no valid audio files.")
                    return

                cues = []
                cur_num, cur_char, lines_buf = None, None, []

                for raw in script_text_area.value.strip().split("\n"):
                    line = raw.strip()
                    if not line:
                        continue

                    m = re.match(r'^(\d+)\s*\.\s*(.+)$', line)
                    if m:
                        if cur_num is not None and lines_buf:
                            spoken = clean_dialogue(" ".join(lines_buf))
                            if spoken:
                                cues.append((cur_num, cur_char, spoken))
                            lines_buf = []
                        cur_num = m.group(1)
                        cur_char = m.group(2).strip().upper()
                        continue

                    if is_first_letter_cipher(line):
                        continue

                    lines_buf.append(line)

                if cur_num is not None and lines_buf:
                    spoken = clean_dialogue(" ".join(lines_buf))
                    if spoken:
                        cues.append((cur_num, cur_char, spoken))

                if not cues:
                    print("❌ Error: No numbered dialogue cues found (e.g., '1.RHODEY').")
                    return

                print(f"🚀 ACTING MODE: Synthesizing {len(cues)} cues for '{clean_folder_name}'...")
                generated = []

                for num, char, dialogue in cues:
                    voice_file = roster.get(char)
                    if not voice_file:
                        for r_name, r_path in roster.items():
                            if char in r_name or r_name in char:
                                voice_file = r_path
                                break

                    if not voice_file:
                        print(f"❌ Skipping cue {num}: '{char}' not found in Voice Roster.")
                        continue

                    out_filename = f"{num}.{char}.wav"
                    out_filepath = os.path.join(work_dir, out_filename)

                    f5tts.infer(
                        ref_file=voice_file,
                        ref_text=univ_ref_text,
                        gen_text=dialogue,
                        file_wave=out_filepath
                    )
                    generated.append(out_filepath)
                    print(f"✔ Created {out_filename}")

                # Packaging into zip with root folder inside
                zip_target = f"/content/{clean_folder_name}.zip"
                with zipfile.ZipFile(zip_target, 'w') as zf:
                    for f in generated:
                        internal_path = os.path.join(clean_folder_name, os.path.basename(f))
                        zf.write(f, arcname=internal_path)

                print("\n" + "="*50)
                print(f"🎉 ALL {len(generated)} SCENE TRACKS PACKAGED!")
                print(f"📁 Internal Folder: {clean_folder_name}/")
                print(f"📦 Downloading {clean_folder_name}.zip directly to your laptop...")
                print("="*50)

                files.download(zip_target)

        finally:
            run_button.description = '▶ GENERATE AUDIO'
            run_button.disabled = False

run_button.on_click(run_synthesis)

# Display the UI
display(widgets.VBox([
    instruction_html,
    mode_dropdown,
    ref_text_input,
    one_audio_box,
    acting_box,
    run_button,
    status_note_html,
    output_console
]))